# Jetstream2: new instance — one-time setup & training commands

Use this notebook as a **checklist**. Run commands on the **remote VM** in a terminal (SSH), not necessarily inside Jupyter unless you start a notebook server on the instance.

**Assumptions**
- You created a Linux VM (e.g. Ubuntu 22.04) on [Jetstream2](https://jetstream-cloud.org/) with a **floating IP** and your **SSH public key** in `~/.ssh/authorized_keys`.
- Default cloud user is often `ubuntu` or `exouser` — replace `USER` and `INSTANCE_IP` below.

---

## 1. From your laptop: SSH into the instance

Replace `INSTANCE_IP` with the VM’s public IP (or hostname) and `USER` with the login user Jetstream gave you.

In [ ]:
# Run locally (Mac/Linux). Windows: use PowerShell or WSL similarly.
ssh USER@INSTANCE_IP

If you use a specific key file:

```bash
ssh -i ~/.ssh/your_jetstream_key.pem USER@INSTANCE_IP
```

**First time:** accept the host key when prompted (`yes`).

---

## 2. On the VM: system packages (once per fresh image)

Ubuntu/Debian example:

In [ ]:
sudo apt update && sudo apt upgrade -y
# python3-full + python3-venv are BOTH required on Ubuntu 22.04/24.04 to create a working venv
# (otherwise `python3 -m venv .venv` produces a broken env that triggers PEP 668 errors).
sudo apt install -y git python3 python3-venv python3-full python3-pip build-essential tmux htop rsync

---

## 3. Project directory and Python venv

Adjust `PROJECT` if you prefer another path. The examples below use `~/topbrain/TopBrain_Algo_Submission` to match our rsync target.

**Always activate the venv** before `python src/train.py`: `source .venv/bin/activate`. If you see `ModuleNotFoundError: No module named 'torch'`, the venv is missing packages — run `pip install -r requirements.txt` again (includes **PyTorch**). For a **GPU** build, after the CPU install works you can reinstall PyTorch using the selector at [pytorch.org](https://pytorch.org/get-started/locally/).

### Troubleshooting venv creation

- **`error: externally-managed-environment` (PEP 668)** when running `pip install ...`: you are not actually inside a venv (or the venv is broken). Recreate it cleanly (see the fixed snippet below). Do **not** use `--break-system-packages`.
- **`No such file or directory: '.venv/bin/python3'`**: the venv is half-created. Delete and recreate with `--copies` to avoid symlink edge cases.

In [ ]:
mkdir -p ~/topbrain && cd ~/topbrain

# Option A: clone from GitHub (HTTPS — needs token/credentials, or SSH — see step 4)
# git clone https://github.com/MahsaAbadian/brain_cta.git TopBrain_Algo_Submission
# cd TopBrain_Algo_Submission

# Option B: rsync from your laptop (see step 5) — then:
cd ~/topbrain/TopBrain_Algo_Submission

# If you already have a broken .venv from a previous attempt, remove it first.
deactivate 2>/dev/null || true
rm -rf .venv

# Use --copies to avoid broken symlinks that cause
# "No such file or directory: '.venv/bin/python3'" on some Jetstream images.
python3 -m venv --copies .venv
source .venv/bin/activate

# Verify you are inside the venv (both paths must be under .../.venv/bin/)
which python
which pip
python -V

# Upgrade pip/setuptools/wheel via the venv's python to avoid PEP 668 issues.
python -m pip install --upgrade pip setuptools wheel
python -m pip install -r requirements.txt

### 3b. Verify GPU / fix PyTorch CUDA driver mismatch

After `pip install -r requirements.txt`, sanity-check the GPU:

```bash
nvidia-smi
python -c "import torch; print(torch.__version__); print('cuda_runtime=', torch.version.cuda); print('cuda_available=', torch.cuda.is_available())"
```

If you see:

```
UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). ...
```

the pip-default PyTorch wheel expects a newer CUDA runtime than the VM driver supports. Install a PyTorch build compiled for CUDA 12.1 (compatible with older drivers on Jetstream2 GPU images):

```bash
pip uninstall -y torch torchvision torchaudio
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

CPU-only fallback (works on any instance, slower):

```bash
pip uninstall -y torch torchvision torchaudio
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
```

> Note: `requirements.txt` in this repo already pins `pandas` conditionally so it builds on Python 3.12 on fresh Jetstream images. No manual patching needed.


---

## 4. (Optional) GitHub over SSH from the VM

Useful if the repo is private or you want to `git pull` without HTTPS tokens.

**On the VM:**

In [ ]:
ssh-keygen -t ed25519 -C "jetstream-vm" -f ~/.ssh/id_ed25519 -N ""
cat ~/.ssh/id_ed25519.pub

Copy the **public** key line and add it in GitHub: **Settings → SSH and GPG keys → New SSH key**.

Test:

In [ ]:
ssh -T git@github.com

Expected: `Hi USERNAME! You've successfully authenticated...`

Then clone with SSH:

```bash
mkdir -p ~/topbrain && cd ~/topbrain
git clone git@github.com:MahsaAbadian/brain_cta.git TopBrain_Algo_Submission
cd TopBrain_Algo_Submission
python3 -m venv --copies .venv && source .venv/bin/activate && python -m pip install --upgrade pip setuptools wheel && python -m pip install -r requirements.txt
```

---

## 5. (Optional) Copy the **code repo** from your laptop with `rsync`

Run **on your laptop** (not on the VM). Replace paths, user, and IP.

This syncs the **whole project folder** (code + whatever data you have locally). For **only** `training_data/` or `training_data_resampled/`, see **section 6** instead.

In [ ]:
# Run on LOCAL machine — example:
 rsync -avz --progress \
  -e "ssh -i ~/.ssh/id_ed25519" \
  /Users/mahsaabadian/MachineLearning/Nazim/TopBrain_Algo_Submission \
   exouser@149.165.151.119:~/topbrain/

After rsync, on the VM:

```bash
cd ~/topbrain/TopBrain_Algo_Submission
python3 -m venv --copies .venv
source .venv/bin/activate
python -m pip install --upgrade pip setuptools wheel
python -m pip install -r requirements.txt
```

---

## 6. Get the TopBrain dataset (download or copy from your machine)

### Where to download

- **Zenodo (TopBrain 2025 release):** [https://zenodo.org/records/16878417](https://zenodo.org/records/16878417) — download the archive(s) listed there (large; use a stable connection).
- **Challenge / documentation:** [https://topbrain2025.grand-challenge.org](https://topbrain2025.grand-challenge.org) — data page and citation details.
- This repo only ships `training_data/README.txt` and `training_data/License.txt`; **scans and labels are not in git**.

### Layout after you unpack (raw training data)

Under the **project root** (same layout as `README.md` **Put the Raw Data Here** in this repo):

- `training_data/imagesTr_topbrain_ct/*.nii.gz`
- `training_data/labelsTr_topbrain_ct/*.nii.gz`
- (optional MR) `training_data/imagesTr_topbrain_mr/`, `training_data/labelsTr_topbrain_mr/`
- `training_data/itksnap_labelmap_txt/` (label maps — keep with the data you use)

### Option A: Download directly on the VM

1. `cd ~/projects/TopBrain_Algo_Submission` (or your project path).
2. Use **wget/curl** with the Zenodo file link(s) from your browser (right‑click the download button → copy URL), or download in the browser on your laptop and use **Option B**.
3. Unzip so the folders above sit under `training_data/`.

### Option B: Copy from your laptop with `rsync` (recommended if you already have the data locally)

Run **on your laptop** (replace paths, user, IP, and optional `-e` SSH key). Sync **into** the project directory on the VM.

**Raw data only** (then preprocess on the VM — step 7):

```bash
rsync -avz --progress \
  -e "ssh -i ~/.ssh/your_key" \
  /path/to/local/TopBrain_Algo_Submission/training_data/ \
  USER@INSTANCE_IP:~/topbrain/TopBrain_Algo_Submission/training_data/
```

**Preprocessed data** (skip long preprocessing on the VM if your laptop already has `training_data_resampled/`):

```bash
rsync -avz --progress \
  -e "ssh -i ~/.ssh/id_ed25519" \
  /Users/mahsaabadian/MachineLearning/Nazim/TopBrain_Algo_Submission \
   exouser@149.165.151.119:~/topbrain/
```

Smaller alternative: **`scp -r`** the same folders (slower than rsync for large datasets).

---

## 7. Data layout the training script expects

Training defaults use **resampled** CTA paths (see `src/train.py` / `src/data_loader.py`):

- `training_data_resampled/imagesTr_topbrain_ct/`
- `training_data_resampled/labelsTr_topbrain_ct/`
- `training_data_resampled/split/` (`train_cases.txt`, `val_cases.txt`)
- `training_data_resampled/itksnap_labelmap_txt/labelmap_topbrain_ct.txt`

If you copied **only raw** `training_data/` to the VM, run **once** from the project root (after `source .venv/bin/activate`):

```bash
python src/preprocess_resample.py --dataset topbrain_ct --copy-metadata
```

That writes into `training_data_resampled/`. If `split/` or labelmaps are missing, copy them from your laptop or re-run split generation per `DOCUMENTATION.md`.

---

## 8. Long runs: `tmux` (recommended)

SSH disconnects will not kill the training job if it runs inside a tmux session.

**On the VM:**

In [ ]:
tmux new -s train

Inside tmux: activate venv, `cd` to project, run training (section 9).

**Detach** (leave job running): `Ctrl+b` then `d`  
**Reattach:** `tmux attach -t train`  
**List sessions:** `tmux ls`

---

## 9. Training command (GPU if available)

From project root with venv activated. Adjust `--out-dir`, epochs, and patch settings to your GPU memory.

In [ ]:
cd ~/topbrain/TopBrain_Algo_Submission
source .venv/bin/activate

# OOM-safe config for ~40 GB GPUs (e.g. A100 40GB on Jetstream2).
# The dominant VRAM cost is clDice (iterative 3D soft-skeletonization across
# all foreground channels at patch 128^3). Tune its flags first, NOT patch size.
#
# Knobs that help if you still OOM, in order of preference:
#   1) PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True  (no accuracy cost)
#   2) --cldice-iters 6                                  (halves recompute graph)
#   3) --cldice-class-ids 1,2,3,4,5,6,7,8,9,10           (restrict channels)
#   4) --base-ch 12                                      (smaller U-Net width)
#   5) --patch-size 96 96 96                             (last resort; cubic cost)

PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
python src/train.py \
  --epochs 200 \
  --lr 1e-3 \
  --num-patches-per-volume 2 \
  --patch-size 128 128 128 \
  --cldice-target-skeleton-dir training_data_resampled/target_skeletons_lee \
  --cldice-iters 6 \
  --out-dir runs/cluster_$(date +%Y%m%d)

Checkpoints and `metrics.csv` appear under `--out-dir`.

**CUDA:** If `nvidia-smi` works, PyTorch should use GPU automatically (`device=cuda` in the script log).

---

## 10. Copy results back to your laptop (optional)

**On your laptop:**

In [ ]:
# rsync -avz USER@INSTANCE_IP:~/topbrain/TopBrain_Algo_Submission/runs/ \
#   ./runs_from_jetstream/

---

## 11. Quick sanity checks on the VM

```bash
cd ~/topbrain/TopBrain_Algo_Submission
source .venv/bin/activate
nvidia-smi
python -c "import torch; print('torch:', torch.__version__, 'cuda_rt:', torch.version.cuda, 'cuda_ok:', torch.cuda.is_available())"
pytest -q
```

If `cuda_ok` is `False` or you see a "driver is too old" warning, see step **3b** to reinstall the cu121 PyTorch wheel.

---

## Notes

- **Security groups / firewall:** OpenStack security groups must allow **SSH (port 22)** from your IP to connect.
- **Floating IP:** If the instance is deleted, the IP may change; update your SSH config and `rsync` targets.
- **Jupyter on the VM:** If you want JupyterLab in the browser, install it in the venv and bind to `0.0.0.0` with a password; **only** do this on a network you trust and lock down the security group to your IP.

For training details, see `TRAINING.md` in the repo root.